In [ ]:
import xml.etree.ElementTree as ET

def extract_tei_heads(file_path, depth=None, first_hi_only=False):
    """
    Extracts text from <head> elements in a TEI XML file.
    Handles nested elements within <head> (collects all descendant text).
    
    Parameters
    ----------
    file_path : str
        Path to the TEI XML file.
    depth : int, optional
        Filter results by this depth value (the @n attribute on <div>).
    first_hi_only : bool, optional
        If True, only extracts the text of the first <hi> child within each <head>.
        If no <hi> child exists, the full <head> text is returned instead.
        
    Returns
    -------
    list of dict
        Each dict has keys: 'depth' and 'text'.
    """

    def get_full_text(element):
        """Recursively collect text from an XML element and its children."""
        text = element.text or ""
        for child in element:
            text += get_full_text(child)
            if child.tail:
                text += child.tail
        return text

    # Parse XML
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Handle TEI namespace if present
    ns = {'tei': 'http://www.tei-c.org/ns/1.0'} if 'http://www.tei-c.org/ns/1.0' in root.tag else {}

    results = []

    # Iterate through all <div> elements
    for div in root.findall(".//tei:div" if ns else ".//div", ns):
        n_attr = div.get('n')
        try:
            div_depth = int(n_attr) if n_attr is not None else None
        except ValueError:
            div_depth = None

        # Skip if filtering by depth
        if depth is not None and div_depth != depth:
            continue

        # Get the first <head> child
        head = div.find("tei:head" if ns else "head", ns)
        if head is not None:
            if first_hi_only:
                # Find first <hi> inside head
                hi = head.find("tei:hi" if ns else "hi", ns)
                if hi is not None:
                    head_text = get_full_text(hi).strip()
                else:
                    # Fallback: no <hi> child found, use full <head> text
                    head_text = get_full_text(head).strip()
            else:
                head_text = get_full_text(head).strip()

            if head_text:
                results.append({
                    "depth": div_depth,
                    "text": head_text
                })

    return results


In [ ]:
# example use

info = []
for o in ['CAP1905', 'HAU1853', 'HOS1879', 'KRA1852', 'KUN1863', 'NAU1858', 'OET1866', 'RIE1905', 'THU1877', 'WEI1860', 'WEI1861']: 
    p = path + o + ".xml"
    all_heads = extract_tei_heads(p, depth=1)
    print("--------------------------------------------------------------")
    print(len(all_heads), o)
    info.append((len(all_heads),o))
    dep = []
    for h in all_heads:
        print(f"Depth {h['depth']}: {h['text']}")
        dep.append(h['depth'])
    print(Counter(dep))